<a href="https://colab.research.google.com/github/n2pakbong/boundary-adapted-pinns/blob/main/colab/periodic_committor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Periodic two-well committor: experiment notebook

This notebook compares the paper's five formulations for the periodic two-well **committor probability** PDE.

**Workflow:** read the equation; run Setup/tests and the tiny preflight on CPU; prepare the finite-difference reference and Monte Carlo labels on CPU; then manually switch to GPU for network training. The longer CPU preparation reports progress and saves to one dated Drive folder. No GPU is requested at startup.

[Experiment guide: full derivation, loss definitions, saved-file map, and references](https://github.com/n2pakbong/boundary-adapted-pinns/blob/da48c2d475896df6afd624789a228fffbbe351f0/experiments/committor/periodic_two_well/README.md).

## 1. The mathematical problem

Consider overdamped Langevin dynamics on the flat torus $\mathbb T^2=[0,1)^2$:

$$
dX_t=-\nabla V(X_t)\,dt+\sqrt{2\varepsilon}\,dW_t \pmod 1,
\qquad
V(x_1,x_2)=a\left[1-\cos\!\left(4\pi(x_1-\tfrac14)\right)\right]
+b\left[1-\cos(2\pi x_2)\right].
$$

Let $A$ and $B$ be two disjoint periodic disks, centered at $(1/4,0)$ and $(3/4,0)$. The **forward committor** is

$$
q(x)=\mathbb P_x(\tau_B<\tau_A),
$$

the probability that a trajectory started at $x$ reaches $B$ before $A$. It solves

$$
\mathcal L q
=-\nabla V\cdot\nabla q+\varepsilon\Delta q=0
\quad\text{in }D=\mathbb T^2\setminus(A\cup B),
$$

with $q=0$ on $\partial A$ and $q=1$ on $\partial B$. Thus this notebook studies a hitting probability, not a mean escape time.

The default experiment uses $a=1$, $b=0.6$, $\varepsilon=0.4$, and periodic chordal radius $r=0.5$. The finite-difference solution is a numerical target for error measurement; it is not exact ground truth.

## 2. What runs, and what uses compute

| Stage | Work | Compute |
|---|---|---|
| Setup | Install code and run tests | CPU |
| Preflight | Tiny one-epoch run of all five methods | CPU |
| Preparation | Finite-difference references and shared Monte Carlo labels | CPU; saved to Drive |
| Training | Five networks and solution-error evaluation | GPU |
| Inspect/evaluate | Saved curves, losses, errors; checkpoint evaluation | CPU |

Leave Colab on CPU through preparation. Only the separately labelled training stage uses GPU. Settings are in `experiments/committor/periodic_two_well/configs/gpu_pilot.json`; this single-seed comparison is exploratory, not a final paper result.

## 3. Setup and CPU tests

Keep Colab on CPU and run the next cell. Enter the exact branch name if using a draft; blank selects `main`. Setup installs the code and runs CPU tests; it does not train.

After CPU preparation, switch to GPU (the runtime restarts), then run Setup again with the same branch. GPU Setup skips CPU tests; the training cell verifies the saved preparation matches the installed revision.

In [ ]:
# CPU setup: select a Git branch, install it, and run mathematical and tiny training tests.
from pathlib import Path
import base64
import os
import subprocess

from google.colab import userdata

CPU_PREFLIGHT_PASSED = False

def run_logged(command, log_path, append=False):
    """Show output and offer the complete log as a download if the command fails."""
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a" if append else "w", encoding="utf-8") as log:
        process = subprocess.Popen(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env={**os.environ, "PYTHONUNBUFFERED": "1"},
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        return_code = process.wait()
    if return_code:
        print(f"\nCommand failed (exit {return_code}). The full traceback is above.")
        try:
            from google.colab import files
            files.download(str(log_path))
            print("Attach the downloaded console.log if you need help diagnosing the error.")
        except Exception as download_error:
            print(f"Log download did not start: {download_error}. Copy the traceback above.")
        raise RuntimeError("Command failed. The traceback and downloadable log are above.")

REPO = Path('/content/boundary-adapted-pinns')
REPO_URL = 'https://github.com/n2pakbong/boundary-adapted-pinns.git'
BRANCH = input('Git branch to install [main]: ').strip() or 'main'

if REPO.exists() and not (REPO / '.git').is_dir():
    raise RuntimeError(f'{REPO} exists but is not a Git clone. Rename that temporary folder, then rerun setup.')

token = None
try:
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    # Publicly readable repositories can be cloned without authentication.
    pass

git_auth = ['git']
if token:
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    git_auth += ['-c', f'http.extraheader=AUTHORIZATION: basic {auth}']

try:
    if not REPO.exists():
        subprocess.run([
            *git_auth, 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO)
        ], check=True)
    else:
        subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', REPO_URL], check=True)
        subprocess.run([
            *git_auth, '-C', str(REPO), 'fetch', '--depth', '1', 'origin', BRANCH
        ], check=True)
        subprocess.run(['git', '-C', str(REPO), 'checkout', '-B', BRANCH, 'FETCH_HEAD'], check=True)
except subprocess.CalledProcessError as error:
    raise RuntimeError(
        'Could not fetch the selected branch. Check the branch name and repository access, then rerun Setup.'
    ) from error
finally:
    del token
    if 'auth' in globals():
        del auth

%cd /content/boundary-adapted-pinns
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
import torch
if torch.cuda.is_available():
    print('GPU runtime detected. Installed the selected branch; CPU tests are not rerun here.')
    print('They must have passed in the CPU runtime before the prepared experiment folder was created.')
else:
    subprocess.run(['python', '-m', 'pytest', '-q'], check=True)
    print('Setup and CPU tests passed. Code revision:', revision)


## 4. CPU preflight: check the complete pipeline without GPU

Run this after Setup passes. It executes all five methods for one epoch with tiny networks, one fixed Monte Carlo anchor, eight short paths, and coarse finite-difference grids. This is a wiring check, not a useful accuracy result.

Its outputs are written under `/content/periodic_committor_preflight`, which is temporary Colab storage. It does not clutter Drive. The cell sets a flag only after it completes successfully; the later GPU cell refuses to start unless this flag is set in the current runtime.

In [ ]:
# CPU only: run the full pipeline in temporary Colab storage.
from pathlib import Path
import shutil

CPU_PREFLIGHT_PASSED = False
preflight_dir = Path('/content/periodic_committor_preflight')
if preflight_dir.exists():
    shutil.rmtree(preflight_dir)
preflight_dir.mkdir()
PREFLIGHT_ROOT = str(preflight_dir)
print('CPU preflight outputs (temporary Colab storage):', PREFLIGHT_ROOT)

run_logged([
    'periodic-committor',
    'experiments/committor/periodic_two_well/configs/cpu_preflight.json',
    '--output-dir', PREFLIGHT_ROOT,
], preflight_dir / 'console.log')
if not (preflight_dir / 'summary.json').is_file():
    raise RuntimeError('CPU preflight produced no summary.json; check the command output above.')
CPU_PREFLIGHT_PASSED = True
print('CPU preflight passed. The full CPU experiment preparation is now available.')

## 5. Boundary conditions and the five trial functions

For each boundary set $j\in\{A,B\}$, the code builds a smooth level-set function $\omega_j$ and a normalized distance-like function $d_j$:

$$
\omega_j(x)=4\sum_{k=1}^2\sin^2\!\bigl(\pi(x_k-c_{jk})\bigr)-r^2,
\qquad
d_j(x)=\frac{\omega_j(x)}
{\sqrt{\omega_j(x)^2+\lvert\nabla\omega_j(x)\rvert^2}}.
$$

Here $c_j$ is the center of set $j$. The boundary is where $\omega_j=0$; in the exterior domain, both $d_A$ and $d_B$ are positive. Define the smooth lift $g$ and the two-boundary factor $\rho$ by

$$
g(x)=\frac{d_A(x)}{d_A(x)+d_B(x)},
\qquad
\rho(x)=\frac{d_A(x)d_B(x)}
{\sqrt{d_A(x)^2+d_B(x)^2}}.
$$

Then $g=0$ on $\partial A$, $g=1$ on $\partial B$, and $\rho=0$ on both boundaries. The three exact-boundary trial functions use

$$
q_\theta(x)=g(x)+\rho(x)^\alpha N_\theta(x).
$$

Boundary-adapted uses $\alpha=1$, over-enforced uses $\alpha=3/2$, and under-enforced uses $\alpha=1/2$. Each satisfies both Dirichlet values by construction. The network output is periodic because the network receives sine and cosine coordinates. For background on the standard PINN residual and exact distance-function boundary enforcement, see Raissi et al. (2019) and Sukumar and Srivastava (2022); the exponent variants listed here are the specific cases in this experiment.

## 6. Which loss belongs to which method?

Every method uses the same interior collocation points and the same Monte Carlo anchor labels within a comparison. Write the interior residual loss and shared solution-value data loss as

$$
\widehat J_{\rm pde}(\theta)
=\frac{1}{N_D}\sum_{i=1}^{N_D}
\left|\mathcal Lq_\theta(x_i)\right|^2,
\qquad
\widehat J_{\rm data}(\theta)
=\frac{1}{N_{\rm data}}\sum_{i=1}^{N_{\rm data}}
\left(q_\theta(z_i)-\widehat q_i\right)^2.
$$

| Method | Trial function | Trained loss components | Boundary treatment |
|---|---|---|---|
| Standard PINN (`simple_pinn`) | $q_\theta=N_\theta$ | PDE, Monte Carlo data, ordinary boundary residual | Samples both boundaries and penalizes the Dirichlet residual. |
| Boundary-adapted | $g+\rho N_\theta$ | PDE, Monte Carlo data | Exact values from the ansatz; no boundary-residual penalty. |
| Over-enforced | $g+\rho^{3/2}N_\theta$ | PDE, Monte Carlo data | Exact values from the ansatz; no boundary-residual penalty. |
| Under-enforced | $g+\rho^{1/2}N_\theta$ | PDE, Monte Carlo data | Exact values from the ansatz; no boundary-residual penalty. |
| Variational/Nitsche | $q_\theta=N_\theta$ | Weak residual, Nitsche stabilization, Monte Carlo data | Weakly imposes the boundary values using Nitsche's method. |

Only the standard PINN has the ordinary boundary-residual component:

$$
\widehat J_{\rm bc}
=\frac{1}{N_A}\sum_i q_\theta(x_i^A)^2
+\frac{1}{N_B}\sum_i\bigl(q_\theta(x_i^B)-1\bigr)^2.
$$

The boundary-adapted, over-enforced, and under-enforced methods have **no extra boundary penalty**. The variational method's stabilization term belongs to the Nitsche weak formulation; it is not applied to an exact-boundary ansatz. The adaptive objective and exact component names are defined in Section 8; the variational method is derived in Section 9. The main experiment guide links each equation to its implementation.

## 7. Monte Carlo labels: the slow CPU stage and its live progress

The committor also has a stochastic representation. Starting at anchor $z_i$, independent Euler–Maruyama trajectories approximate the Langevin dynamics:

$$
X_{n+1}
=\left[X_n-\nabla V(X_n)\Delta t
+\sqrt{2\varepsilon\Delta t}\,\xi_n\right]\pmod 1,
\qquad \xi_n\sim\mathcal N(0,I_2).
$$

This carries over the pointwise solution-value regularizer used in the mean-escape-time notebooks: here the target value is the committor probability, and it complements rather than replaces the PDE residual. Each trajectory stops on first entering $A$ or $B$, or when it reaches the configured step cap. The label is the fraction of resolved trajectories that hit $B$ first:

$$
\widehat q_i
=\frac{\#\{\text{resolved trajectories from }z_i\text{ that hit }B\text{ first}\}}
{\#\{\text{resolved trajectories from }z_i\}}.
$$

Capped, unresolved paths are **not** counted as hits of $A$. Training stops with an error if an anchor's resolved fraction is below the configured minimum. The reported binomial standard error is conditional on the resolved paths; if paths remain unresolved, the guide explains the possible censoring bias and time-step error.

**What you will see while it runs:** the console reports the current step against the configured cap, paths that have hit either set, anchors meeting the required resolution, minimum and mean anchor resolution, and elapsed time. It prints the completed resolution summary before training begins. The same anchors and labels are saved once and shared across the five methods in `monte_carlo_labels.npz` and `monte_carlo_labels.json`. Neural-network approaches to committor computation are discussed, for example, by Khoo, Lu, and Ying (2018) and Song, Cameron, and Yang (2023).

## 8. Trainable loss balancing: the objective and the reported values

The paper's training notebooks use a trainable scale for each active loss. For component $k$, let $s_k=\log\sigma_k$. The objective implemented here is

$$
\mathcal J(\theta,s)
=\sum_{k\in\mathcal C}\left(e^{-2s_k}\widehat J_k(\theta)+s_k\right),
$$

where $\mathcal C$ contains only the components listed for that method in the table above. Written by method, the standard PINN uses

$$
\mathcal J_{\rm standard}
=e^{-2s_{\rm pde}}\widehat J_{\rm pde}+s_{\rm pde}
+e^{-2s_{\rm data}}\widehat J_{\rm data}+s_{\rm data}
+e^{-2s_{\rm bc}}\widehat J_{\rm bc}+s_{\rm bc}.
$$

Each exact-boundary model uses only the PDE and Monte Carlo data terms:

$$
\mathcal J_{\rm exact}
=e^{-2s_{\rm pde}}\widehat J_{\rm pde}+s_{\rm pde}
+e^{-2s_{\rm data}}\widehat J_{\rm data}+s_{\rm data}.
$$

The variational model uses the weak residual, Nitsche stabilization, and the same Monte Carlo data term:

$$
\mathcal J_{\rm Nitsche}
=e^{-2s_{\rm weak}}\widehat J_{\rm weak}+s_{\rm weak}
+e^{-2s_{\rm stab}}\widehat J_{\rm stab}+s_{\rm stab}
+e^{-2s_{\rm data}}\widehat J_{\rm data}+s_{\rm data}.
$$

In code, $s_k$ is the matching trainable parameter named `log_sigma_*`, and the effective weight is $e^{-2s_k}$. This is the no-one-half convention used by the original training notebooks. The trainable-scale construction follows Kendall, Gal, and Cipolla (2018).

Every 20 epochs, training prints the epoch, total optimized objective, and active **raw** loss components. Each record also saves weighted components and current effective weights. `metrics.jsonl` contains the records, `losses.png` plots the objective and loss histories, and `final_training_metrics.json` plus `summary.json` save the final components. The optimized objective includes the trainable $s_k$ offsets and can be negative; each raw mean-square loss remains nonnegative. Solution accuracy is reported separately against the finite-difference reference.

## 9. Variational/Nitsche method: what its boundary term means

Because the drift is a gradient, set $m(x)=e^{-V(x)/\varepsilon}$. The strong committor equation is equivalent to the divergence-form equation

$$
\mathcal Lq=0
\quad\Longleftrightarrow\quad
\nabla\cdot\left(\varepsilon m\nabla q\right)=0.
$$

The tests are periodic Gaussian functions centered at points $c_j$ in $D$:

$$
\phi_j(x)=\exp\!\left[
-\frac{4\sum_{k=1}^2\sin^2\!\bigl(\pi(x_k-c_{jk})\bigr)}{2\ell^2}
\right].
$$

For each test function, integration by parts gives a volume contribution, a normal-flux contribution, and the symmetric Nitsche contribution:

$$
\begin{aligned}
R_j={}&\varepsilon\int_D m\nabla q_\theta\cdot\nabla\phi_j\,dx\\
&-\varepsilon\int_{\partial D}m\,\partial_{n_D}q_\theta\,\phi_j\,ds\\
&-\varepsilon\int_{\partial D}m(q_\theta-g)\,\partial_{n_D}\phi_j\,ds.
\end{aligned}
$$

Here $n_D$ is the outward normal of the exterior domain $D$, pointing into the removed sets. The second line is the flux term; the third is the symmetric Nitsche term. The stabilization moment is

$$
P_j=\frac{\gamma\varepsilon}{h}
\int_{\partial D}m(q_\theta-g)\phi_j\,ds.
$$

The two trained components are their mean squares over the $N_\phi$ test functions:

$$
\widehat J_{\rm weak}=\frac{1}{N_\phi}\sum_{j=1}^{N_\phi}R_j^2,
\qquad
\widehat J_{\rm stab}=\frac{1}{N_\phi}\sum_{j=1}^{N_\phi}P_j^2.
$$

They are logged as `nitsche_weak_loss` and `nitsche_penalty_loss`. The second name refers to Nitsche stabilization, not the standard PINN's sampled boundary residual. Neither component is applied to the three exact-boundary models. The weak boundary treatment follows the variational PINN/Nitsche literature (including Berrone et al., 2023); the experiment guide documents the quadrature and boundary orientation.

**Check the equations in code:** the PDE and boundary geometry are in `src/boundary_pinn/problems/committor/periodic_two_well.py`; Monte Carlo and weak-form integrals are in `paper_methods.py`; ansatzes, active loss sets, adaptive scales and epoch logs are in `training.py`; orchestration and saved error tables are in `comparison.py`.

## 10. Optional: clean up old Drive folders before a new run

This is optional and does not affect the code or results. Earlier notebook versions may have left timestamped CPU preflight folders or failed pilot folders in Drive. Run the next cell if you want to remove those old folders.

It first prints the exact candidate paths. It excludes any GPU folder containing checkpoints, references, plots, or other result files. **Nothing is deleted unless you type `DELETE` at the prompt.** Completed runs are preserved. No new preflight or experiment outputs are written to Drive by the CPU preflight; new GPU results are written to a new dated run folder.

In [ ]:
# Optional cleanup: list eligible old folders, then require an explicit confirmation.
from pathlib import Path
import re
import shutil
from google.colab import drive

drive.mount('/content/drive')
old_run_root = Path('/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor')
known_startup_files = {'console.log', 'comparison_config.json', 'comparison_metadata.json'}
cleanup_candidates = []
if old_run_root.is_dir():
    for folder in sorted(old_run_root.iterdir()):
        if not folder.is_dir():
            continue
        if re.fullmatch(r'preflight_\d{8}_\d{6}', folder.name):
            cleanup_candidates.append(folder)
        elif re.fullmatch(r'periodic_committor_\d{8}_\d{6}', folder.name):
            children = list(folder.iterdir())
            if children and all(child.is_file() and child.name in known_startup_files for child in children):
                cleanup_candidates.append(folder)

if not cleanup_candidates:
    print('No eligible old preflight or empty failed-pilot folders were found. Nothing was deleted.')
else:
    print('Eligible folders (completed runs and folders containing checkpoints/results are excluded):')
    for folder in cleanup_candidates:
        print(' -', folder)
    confirmation = input('To permanently delete exactly these listed folders, type DELETE: ').strip()
    if confirmation == 'DELETE':
        for folder in cleanup_candidates:
            shutil.rmtree(folder)
            print('Deleted:', folder)
        print(f'Deleted {len(cleanup_candidates)} listed folders.')
    else:
        print('No folders deleted.')


## 11. CPU experiment preparation: reference grids and Monte Carlo labels

After Setup and preflight, run this with a CPU runtime. It solves both finite-difference grids and generates shared Feynman–Kac labels, with grid timings and live path progress. It does **not** train a network or use GPU.

It saves the configuration, code revision, reference, labels, resolution data, and console log in one dated Drive folder. Keep that folder for GPU training. If Drive is already mounted, continue.

In [ ]:
# CPU only: prepare the reference and common labels before allocating a GPU.
from datetime import datetime
from pathlib import Path
import json
import subprocess
import torch
from google.colab import drive

if not globals().get('CPU_PREFLIGHT_PASSED', False):
    raise RuntimeError('CPU preparation blocked: run Setup and pass the CPU preflight in this CPU runtime first.')

torch.set_num_threads(2)
if torch.cuda.is_available():
    raise RuntimeError(
        'A GPU runtime is attached. Stop here and change Runtime > Change runtime type to CPU. '
        'This potentially long preparation stage must finish before allocating a GPU.'
    )

drive.mount('/content/drive')
run_name = 'periodic_committor_' + datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_ROOT = f'/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor/{run_name}'
Path(RUN_ROOT).mkdir(parents=True, exist_ok=False)
print('CPU preparation folder (keep this path for the GPU stage):', RUN_ROOT, flush=True)

run_logged([
    'periodic-committor',
    'experiments/committor/periodic_two_well/configs/gpu_pilot.json',
    '--output-dir', RUN_ROOT,
    '--prepare-cpu-only',
], Path(RUN_ROOT) / 'console.log')
prep_path = Path(RUN_ROOT, 'cpu_preparation.json')
if not prep_path.is_file():
    raise RuntimeError('CPU preparation ended without cpu_preparation.json; check the console output above.')
prep_record = json.loads(prep_path.read_text(encoding='utf-8'))
preflight_revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
if prep_record.get('preparation_git_revision') != preflight_revision:
    raise RuntimeError('CPU preflight and prepared files came from different code revisions. Repeat Setup and CPU preparation from one branch revision.')
prep_record['cpu_preflight_passed'] = True
prep_record['cpu_preflight_git_revision'] = preflight_revision
prep_path.write_text(json.dumps(prep_record, indent=2) + '\n', encoding='utf-8')
print('CPU preparation is complete. No model training has run.', flush=True)

## 12. GPU training: use the prepared folder, without repeating CPU work

Use GPU only after CPU preparation is complete and saved on Drive:

1. Choose **Runtime → Change runtime type → GPU**; Colab restarts.
2. Run Setup with the same branch. Do not rerun preflight or preparation.
3. Run the next cell and paste the Drive folder printed during preparation.

The cell checks the preparation marker and repository revision, then loads the saved reference and labels. It trains five networks for 6,000 epochs each, logs every 20 epochs, and saves checkpoints in the same folder. It does not repeat CPU work. This is exploratory, single-seed training.

In [ ]:
# GPU required: train only; the finite-difference reference and Monte Carlo labels are already on Drive.
from pathlib import Path
import json
import subprocess
import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU is attached. Choose Runtime > Change runtime type > GPU, reconnect, '
        'run Setup to install this branch, then run this cell. Do not repeat CPU preparation.'
    )
drive.mount('/content/drive')
RUN_ROOT = input('Paste the CPU preparation folder path printed in Section 11: ').strip()
prep_path = Path(RUN_ROOT, 'cpu_preparation.json')
if not prep_path.is_file():
    raise FileNotFoundError(
        f'No completed CPU preparation was found in {RUN_ROOT}. '
        'Return to the CPU runtime and finish Section 11 first.'
    )
if not Path(RUN_ROOT, 'comparison_config.json').is_file():
    raise FileNotFoundError(f'The prepared folder is missing its configuration: {RUN_ROOT}')
prep_record = json.loads(prep_path.read_text(encoding='utf-8'))
if prep_record.get('cpu_preflight_passed') is not True:
    raise RuntimeError('This folder was not prepared after a passing CPU preflight. Return to the CPU runtime and complete Sections 3, 4, and 11.')
current_revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
if prep_record.get('cpu_preflight_git_revision') != current_revision:
    raise RuntimeError('This prepared folder belongs to a different code revision. Repeat CPU setup, preflight, and preparation using the same branch revision.')
print('GPU training outputs will be saved in:', RUN_ROOT, flush=True)
run_logged([
    'periodic-committor',
    'experiments/committor/periodic_two_well/configs/gpu_pilot.json',
    '--output-dir', RUN_ROOT,
    '--train-prepared',
], Path(RUN_ROOT) / 'console.log', append=True)

## 13. Inspect the completed run

Run after training. This cell uses `RUN_ROOT` and does not train. It displays the finite-difference grid check; absolute and relative $L^2$, $H^1$, $H^1$-seminorm, $H^2$, and $H^2$-seminorm errors; final raw loss components; and saved plots.

Errors use the finest configured finite-difference solution. Grid-to-grid differences estimate reference-resolution sensitivity, not an error bound. Figures are previews; arrays, metrics, and checkpoints are saved in the printed Drive folder.

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display
import pandas as pd

with open(f'{RUN_ROOT}/summary.json') as handle:
    summary = json.load(handle)

method_summaries = summary.get('methods', {})
accuracy_fields = [
    'grid_l2_abs', 'grid_l2_rel',
    'grid_h1_abs', 'grid_h1_rel',
    'grid_h1_seminorm_abs', 'grid_h1_seminorm_rel',
    'grid_h2_abs', 'grid_h2_rel',
    'grid_h2_seminorm_abs', 'grid_h2_seminorm_rel',
]
if method_summaries and all(
    all(field in metrics for field in accuracy_fields)
    for metrics in method_summaries.values()
):
    display(pd.DataFrame(method_summaries).T[accuracy_fields])
else:
    print(
        'This run summary predates the full Sobolev error report. '
        'Run Step 5 to calculate absolute and relative L2, H1, and H2 errors '
        'from its saved checkpoints on CPU; no GPU retraining is needed.'
    )

print('Final raw training losses (not solution-error metrics):')
training_rows = {}
for method, metrics in method_summaries.items():
    training = metrics.get('final_training_metrics', {})
    raw_components = training.get('raw_loss_components', {})
    if not raw_components:
        metrics_path = Path(RUN_ROOT) / method / 'metrics.jsonl'
        if metrics_path.is_file():
            lines = [line for line in metrics_path.read_text().splitlines() if line.strip()]
            if lines:
                last = json.loads(lines[-1])
                raw_components = {
                    key: value for key, value in last.items()
                    if key.endswith('_loss') or key.endswith('_mse')
                }
                training = {
                    'final_training_objective': last.get('objective', last.get('loss')),
                    'raw_loss_components': raw_components,
                }
    training_rows[method] = {
        'final objective': training.get('final_training_objective'),
        **raw_components,
    }
display(pd.DataFrame(training_rows).T)

print('Finite-difference grid comparison:')
print(json.dumps(summary.get('reference_convergence', {}), indent=2))
display(Image(filename=f'{RUN_ROOT}/losses.png'))
for method in method_summaries:
    figure = Path(RUN_ROOT) / f'committor_{method}.png'
    if figure.is_file():
        display(Image(filename=str(figure)))

## 14. Re-evaluate saved checkpoints on CPU

This separate cell reloads each method's `best.pt` checkpoint and recomputes errors against the saved finite-difference reference. It uses CPU only, does not retrain, and does not overwrite the original `summary.json`. It displays the Sobolev errors and final raw training-loss components and writes `saved_model_evaluation.json` to the same run folder.

If using a fresh Colab runtime, paste the path printed by the GPU run when prompted.

In [ ]:
# CPU only: compare saved best.pt checkpoints with the FDM target; this does not train.
from pathlib import Path
import json
import subprocess
from google.colab import drive
from IPython.display import display
import pandas as pd

if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')

if 'RUN_ROOT' not in globals() or not Path(RUN_ROOT, 'comparison_config.json').is_file():
    RUN_ROOT = input('Paste the completed GPU pilot Drive folder path: ').strip()

if not Path(RUN_ROOT, 'comparison_config.json').is_file():
    raise FileNotFoundError(f'No comparison_config.json found in {RUN_ROOT}')

command = ['periodic-committor', '--output-dir', str(RUN_ROOT), '--evaluate-saved-models']
subprocess.run(command, check=True)

report_path = Path(RUN_ROOT) / 'saved_model_evaluation.json'
report = json.loads(report_path.read_text(encoding='utf-8'))
print('Training code revision:', report.get('training_git_revision') or 'not recorded for this older run')
print('FDM reference:', report['reference_file'])
display_fields = [
    'grid_l2_abs', 'grid_l2_rel',
    'grid_h1_abs', 'grid_h1_rel',
    'grid_h1_seminorm_abs', 'grid_h1_seminorm_rel',
    'grid_h2_abs', 'grid_h2_rel',
    'grid_h2_seminorm_abs', 'grid_h2_seminorm_rel',
    'source_boundary_rmse', 'target_boundary_rmse',
]
display(pd.DataFrame(report['methods']).T[display_fields])

print('Final training-loss components from the saved run:')
loss_rows = {}
for method, result in report['methods'].items():
    training = result.get('final_training_metrics', {})
    loss_rows[method] = {
        'final objective': training.get('final_training_objective'),
        **training.get('raw_loss_components', {}),
        'MC data term used': training.get('monte_carlo_labels') is not None,
    }
display(pd.DataFrame(loss_rows).T)
print('Full saved report:', report_path)

## 15. Resume an interrupted run

Use only for an interrupted GPU run. If Colab restarted, choose GPU and run Setup with the same branch used for CPU preparation. Paste the saved Drive folder path; the runner checks its revision and resumes each method from `last.pt`. Do not repeat preflight or preparation.

For a new experiment, use the GPU training section to create a new dated folder.

In [ ]:
# Resume only a GPU run that stopped before completion.
from pathlib import Path
import json
import subprocess
import torch
from google.colab import drive

if not torch.cuda.is_available():
    raise RuntimeError('Resume requires a GPU runtime. Change the runtime type to GPU, reconnect, and run Setup first.')
drive.mount('/content/drive')
RUN_ROOT = input('Paste the interrupted run folder path printed at the start of training: ').strip()
prep_path = Path(RUN_ROOT, 'cpu_preparation.json')
if not prep_path.is_file():
    raise FileNotFoundError('That folder has no completed CPU preparation. Paste the run folder path from this notebook.')
prep_record = json.loads(prep_path.read_text(encoding='utf-8'))
current_revision = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
if prep_record.get('cpu_preflight_passed') is not True or prep_record.get('cpu_preflight_git_revision') != current_revision:
    raise RuntimeError('This run folder does not match the passed CPU preflight and installed code revision.')

run_logged([
    'periodic-committor',
    'experiments/committor/periodic_two_well/configs/gpu_pilot.json',
    '--output-dir', RUN_ROOT,
    '--train-prepared',
    '--resume',
], Path(RUN_ROOT) / 'console.log', append=True)
print('The interrupted GPU run has resumed and completed.')

## References

- M. Raissi, P. Perdikaris, and G. E. Karniadakis, “Physics-informed neural networks,” *Journal of Computational Physics* 378 (2019), 686–707. [doi:10.1016/j.jcp.2018.10.045](https://doi.org/10.1016/j.jcp.2018.10.045).
- N. Sukumar and A. Srivastava, “Exact imposition of boundary conditions with distance functions in physics-informed deep neural networks,” *Computer Methods in Applied Mechanics and Engineering* 389 (2022), 114333. [doi:10.1016/j.cma.2021.114333](https://doi.org/10.1016/j.cma.2021.114333).
- S. Berrone, C. Canuto, M. Pintore, and N. Sukumar, “Enforcing Dirichlet boundary conditions in physics-informed neural networks and variational physics-informed neural networks,” *Heliyon* 9(8) (2023), e18820. [doi:10.1016/j.heliyon.2023.e18820](https://doi.org/10.1016/j.heliyon.2023.e18820).
- A. Kendall, Y. Gal, and R. Cipolla, “Multi-task learning using uncertainty to weigh losses for scene geometry and semantics,” *CVPR* (2018). [paper](https://openaccess.thecvf.com/content_cvpr_2018/html/Kendall_Multi-Task_Learning_Using_CVPR_2018_paper.html).
- Y. Khoo, J. Lu, and L. Ying, “Solving for high dimensional committor functions using artificial neural networks,” [arXiv:1802.10275](https://arxiv.org/abs/1802.10275) (2018).
- Z. Song, M. K. Cameron, and H. Yang, “A Finite Expression Method for Solving High-Dimensional Committor Problems,” [arXiv:2306.12268](https://arxiv.org/abs/2306.12268) (2023).
- N. Tepakbong, J. Fan, X. Zhou, and D.-X. Zhou, “Boundary-Adapted PINNs for Elliptic Dirichlet Problems,” [arXiv:2607.19167](https://arxiv.org/abs/2607.19167) (2026).